# Ambiance — Model Evaluation (Google Colab)
**Phase 5:** Measure the trained model's real-world performance on the held-out test set.

Outputs:
- Classification report — precision, recall, F1 per class
- Confusion matrix heatmap
- Top-5 accuracy
- Sample misclassified images (helps diagnose *which* styles the model confuses)

**Before running:** Confirm `ambiance_model.h5` exists in `MyDrive/dl-project/` from Phase 4.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

DRIVE_DIR  = '/content/drive/MyDrive/dl-project'
TEST_DIR   = '/content/dataset_test'
MODEL_PATH = os.path.join(DRIVE_DIR, 'ambiance_model.h5')
HISTORY_PATH = os.path.join(DRIVE_DIR, 'training_history.json')

print("GPU detected:     ", tf.config.list_physical_devices('GPU'))
print("Model exists:     ", os.path.exists(MODEL_PATH))
print("History exists:   ", os.path.exists(HISTORY_PATH))

In [ ]:
import shutil

if not os.path.exists(TEST_DIR):
    print("Copying test set to local Colab storage...")
    shutil.copytree('/content/drive/MyDrive/dl-project/dataset_test', TEST_DIR)
    print("Done.")
else:
    print("Test set already on local storage.")

## 1. Build the test generator

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# No augmentation — we evaluate on unmodified images so metrics reflect real-world performance.
# preprocess_input must match exactly what was used during training; using a different
# normalisation here would produce a train/test domain mismatch and understate true accuracy.
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   # order must stay fixed so predictions align with true labels
)

CLASS_NAMES = list(test_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Test images: {test_gen.samples}  |  Classes: {NUM_CLASSES}")
print("Classes:", CLASS_NAMES)

## 2. Load the trained model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
print(f"Model loaded from: {MODEL_PATH}")

trainable     = sum(p.numpy().size for p in model.trainable_weights)
non_trainable = sum(p.numpy().size for p in model.non_trainable_weights)
print(f"Trainable params:     {trainable:,}")
print(f"Non-trainable params: {non_trainable:,}")

## 3. Run predictions on the full test set
We run the model over every test image once and collect the predicted class for each.
This is the only unbiased evaluation — these images were never seen during training or validation.

In [ ]:
# predict() returns a (num_samples, 19) probability matrix.
# steps must cover all images — ceil(samples / batch_size).
steps = -(-test_gen.samples // BATCH_SIZE)   # ceiling division
y_prob = model.predict(test_gen, steps=steps, verbose=1)

# Predicted class = index of the highest probability
y_pred = np.argmax(y_prob, axis=1)

# True labels from the generator (integers, not one-hot)
y_true = test_gen.classes

# Trim to exact sample count (last batch may be padded)
y_pred = y_pred[:test_gen.samples]
y_true = y_true[:test_gen.samples]

overall_acc = np.mean(y_pred == y_true)
print(f"\nOverall test accuracy: {overall_acc*100:.1f}%")

## 4. Classification report — precision, recall, F1 per class
**Precision** = of all times the model predicted class X, how often was it correct?

**Recall** = of all actual class X images, how many did the model catch?

**F1** = harmonic mean of precision and recall — the single most useful per-class metric when classes may be imbalanced.

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print(report)

# Also save to Drive so we can reference it in the presentation without re-running
report_path = os.path.join(DRIVE_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(report)
print(f"Saved to: {report_path}")

## 5. Per-class F1 bar chart
Visualises which styles are easy vs hard to classify. Low F1 classes are where styles look similar
(e.g. Contemporary vs Transitional vs Modern all share clean lines and neutral palettes).

In [ ]:
from sklearn.metrics import f1_score

f1_per_class = f1_score(y_true, y_pred, average=None)
macro_f1     = f1_score(y_true, y_pred, average='macro')

# Sort from lowest to highest F1 so weak classes are easy to spot
order = np.argsort(f1_per_class)
sorted_names = [CLASS_NAMES[i] for i in order]
sorted_f1    = f1_per_class[order]

colours = ['#d62728' if v < 0.4 else '#ff7f0e' if v < 0.6 else '#2ca02c' for v in sorted_f1]

plt.figure(figsize=(14, 6))
bars = plt.barh(sorted_names, sorted_f1, color=colours)
plt.axvline(x=macro_f1, color='steelblue', linestyle='--', label=f'Macro F1 = {macro_f1:.3f}')
plt.xlabel('F1 Score')
plt.title('Per-class F1 Score (red < 0.4 · orange < 0.6 · green ≥ 0.6)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'f1_per_class.png'), dpi=150)
plt.show()

## 6. Confusion matrix
Each cell (i, j) counts how many images of true class i were predicted as class j.
Strong diagonal = good. Off-diagonal clusters reveal which styles the model systematically confuses.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# Normalise each row to percentages so class size differences don't dominate the colours
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm_norm,
    annot=True, fmt='.2f',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cmap='Blues',
    linewidths=0.3,
    ax=ax
)
ax.set_xlabel('Predicted class', fontsize=12)
ax.set_ylabel('True class', fontsize=12)
ax.set_title('Normalised Confusion Matrix (row = true class, values = fraction predicted)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 7. Top-5 accuracy
Top-5 accuracy asks: was the correct class anywhere in the model's 5 highest-probability predictions?
For a 19-class problem with visually similar styles (Contemporary / Transitional / Modern),
top-5 accuracy gives a fairer picture of how well the model understands the style space.

In [ ]:
# top-5 indices for each test image (descending probability)
top5_indices = np.argsort(y_prob[:test_gen.samples], axis=1)[:, -5:]

top5_correct = sum(
    y_true[i] in top5_indices[i]
    for i in range(test_gen.samples)
)
top5_acc = top5_correct / test_gen.samples

print(f"Top-1 accuracy: {overall_acc*100:.1f}%")
print(f"Top-5 accuracy: {top5_acc*100:.1f}%")
print(f"Macro F1:       {macro_f1:.3f}")

## 8. Sample misclassified images
Shows images the model got wrong alongside its top-3 predictions.
Useful for diagnosing *why* certain styles are hard — often the model is not "wrong" in an obvious way
(e.g. a very neutral Transitional room being called Contemporary is an understandable mistake).

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Collect file paths from the generator
file_paths = test_gen.filepaths

# Find indices where the prediction was wrong
wrong_idx = np.where(y_pred != y_true)[0]
np.random.seed(42)
sample_idx = np.random.choice(wrong_idx, size=min(12, len(wrong_idx)), replace=False)

fig, axes = plt.subplots(3, 4, figsize=(18, 14))
for ax, idx in zip(axes.flat, sample_idx):
    img = load_img(file_paths[idx], target_size=IMG_SIZE)
    ax.imshow(img)

    # Top-3 predictions with probabilities
    top3 = np.argsort(y_prob[idx])[-3:][::-1]
    pred_text = '\n'.join(
        f"{CLASS_NAMES[c]} ({y_prob[idx][c]*100:.0f}%)" for c in top3
    )
    true_label = CLASS_NAMES[y_true[idx]]
    ax.set_title(f"True: {true_label}\n{pred_text}", fontsize=7)
    ax.axis('off')

plt.suptitle('Misclassified examples — true label (top) vs model\'s top-3 predictions', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'misclassified_samples.png'), dpi=150)
plt.show()

print(f"Total misclassified: {len(wrong_idx)} / {test_gen.samples} ({len(wrong_idx)/test_gen.samples*100:.1f}%)")

## 9. Training curves (from saved history)
Reloads the history saved at the end of Phase 4 to verify the model trained cleanly —
no divergence, no extreme overfitting (train accuracy far above val accuracy).

In [ ]:
with open(HISTORY_PATH) as f:
    h = json.load(f)

epochs = range(1, len(h['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, h['loss'],     label='Train loss')
axes[0].plot(epochs, h['val_loss'], label='Val loss')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()

axes[1].plot(epochs, h['accuracy'],     label='Train accuracy')
axes[1].plot(epochs, h['val_accuracy'], label='Val accuracy')
axes[1].set_title('Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Training curves — Phase 4b (fine-tuned)', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Final results summary

In [ ]:
best_class  = CLASS_NAMES[np.argmax(f1_per_class)]
worst_class = CLASS_NAMES[np.argmin(f1_per_class)]

print("── Evaluation Results ──────────────────────────────")
print(f"Test images evaluated:  {test_gen.samples}")
print(f"Top-1 accuracy:         {overall_acc*100:.1f}%")
print(f"Top-5 accuracy:         {top5_acc*100:.1f}%")
print(f"Macro F1:               {macro_f1:.3f}")
print(f"Best class  (F1):       {best_class}  ({np.max(f1_per_class):.3f})")
print(f"Worst class (F1):       {worst_class}  ({np.min(f1_per_class):.3f})")
print()
print("Saved artefacts:")
for name in ['classification_report.txt', 'f1_per_class.png', 'confusion_matrix.png', 'misclassified_samples.png']:
    print(f"  {os.path.join(DRIVE_DIR, name)}")
print()
print("Next step: app.py — Streamlit frontend for the live demo.")